# First KPI Lookup

A beginner-tier walkthrough of `mineproductivity.kpis` -- the first notebook in the Learning & Benchmark Suite v1.0's pedagogical progression. No prior MineProductivity knowledge assumed.

By the end of this notebook you will have:

1. Loaded a small, realistic one-shift sample dataset into an `EventStore`.
2. Computed `PROD.TPH` (Tonnes Per Hour) -- the platform's headline production KPI.
3. Inspected a `KPIResult`'s provenance (unit, row count, warnings), not just its bare number.
4. Discovered what other KPIs are available and what each one means, without reading any source code.

In [1]:
# If you're running this outside the repository's own dev environment:
# %pip install -e "..[analytics]"
import sys
from pathlib import Path

REPO_ROOT = (
    Path.cwd().resolve().parents[1]
    if (Path.cwd() / "pyproject.toml").exists() is False
    else Path.cwd()
)
sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "examples" / "kpis"))
print(f"repo root: {REPO_ROOT}")

repo root: C:\MineProductivity


## 1. Load the sample dataset

`examples/kpis/_dataset.py` parses six CSV fixtures under `tests/fixtures/kpis/` -- haul cycles, downtime, a production reading, fuel burn, delays, and a safety observation -- all for one shift, `A-2026-06-25` (06:00-18:00 UTC), and appends them as canonical events to a real `EventStore`. This notebook reuses that exact loader, so the numbers below match `examples/kpis/01_simple_execution.py`.

In [2]:
from _dataset import SHIFT_ID, build_engine

engine = build_engine()
print(f"Engine built for shift {SHIFT_ID!r}")

Engine built for shift 'A-2026-06-25'


## 2. Compute PROD.TPH

`PROD.TPH` is the design specification's own reference exemplar: payload moved per operating hour, `sum(payload_t) / sum(operating_h)`. `KPIEngine.execute()` resolves the KPI's dependency graph (none, here -- it's a leaf), scans the event store for exactly the rows it needs, and returns a `Result` wrapping a `KPIResult`.

In [3]:
result = engine.execute("PROD.TPH", window="shift", scope={"shift": SHIFT_ID})
assert result.is_ok
tph = result.unwrap()
print(f"PROD.TPH = {tph.value:.1f} {tph.unit}")

PROD.TPH = 216.8 t/h


## 3. A KPIResult is more than a number

Every computation carries its own provenance: how many rows it was computed from (`n`), and any warnings (e.g. missing data) -- never a silent `NaN` or a crash.

In [4]:
print(f"code:     {tph.code}")
print(f"value:    {tph.value}")
print(f"unit:     {tph.unit}")
print(f"n (rows): {tph.n}")
print(f"warnings: {tph.warnings}")

# Exporting to a DataFrame is one method call away:
tph.to_frame()

code:     PROD.TPH
value:    216.8292682926829
unit:     t/h
n (rows): 12
warnings: ()


,code,value,unit,n
0,PROD.TPH,216.829268,t/h,12


### What happens with no data?

Asking for a shift that doesn't exist in the store never raises -- it returns a `KPIResult` with `value=None`, the "qualify, don't coerce" stance every KPI in the Standard Library follows.

In [5]:
empty = engine.execute("PROD.TPH", window="shift", scope={"shift": "no-such-shift"}).unwrap()
print(f"value={empty.value!r} warnings={empty.warnings}")

value=None warnings=()


## 4. Discover what else is available

Every KPI is a discoverable, self-describing object (`KPI-as-object`) -- `REGISTRY` lists every registered code, and each one's `.meta` carries its full governed metadata: business purpose, formula, unit, and more.

In [6]:
from mineproductivity.kpis import REGISTRY

print(f"{len(REGISTRY)} KPIs registered:")
for code_name in sorted(REGISTRY):
    print(f"  {code_name}")

12 KPIs registered:
  COST.FuelPerTonne
  DISP.TotalDelayHours
  ENERGY.FuelConsumed
  HAUL.TruckCycleTime
  MAINT.MTTR
  PROD.TPH
  QUAL.OreProportion
  SAFE.SpeedViolationCount
  UTIL.OEE
  UTIL.PA
  UTIL.Performance
  UTIL.UA


In [7]:
meta = REGISTRY.get("HAUL.TruckCycleTime").meta
print(f"{meta.code} -- {meta.official_name}")
print(f"  business purpose:     {meta.business_purpose}")
print(f"  operational question: {meta.operational_question}")
print(f"  formula:              {meta.formula}")

HAUL.TruckCycleTime -- Truck Cycle Time
  business purpose:     Cycle time is the fundamental unit haulage throughput and match-factor analysis build on.
  operational question: How long, on average, does one haul cycle take on this route/fleet?
  formula:              mean(queue_min + spot_min + load_min + haul_min + dump_min + return_min)


## Next steps

- `examples/kpis/02_composite_oee.py` -- how a composite KPI (`UTIL.OEE`) resolves its own dependency graph before combining results.
- `examples/kpis/03_batch_summary.py` -- computing many KPIs in a single event-store scan.
- `examples/kpis/04_discovery.py` -- filtering `REGISTRY` by namespace and by composite-vs-leaf.
- `src/mineproductivity/kpis/README.md` -- the full package architecture and extension guide.